In [1]:
import os

os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]="2"

In [2]:
from setproctitle import setproctitle
setproctitle("ddqn_vs_mcts")

In [3]:
import sys
sys.path.append('..')

In [4]:
import tensorflow as tf
gpus = tf.config.experimental.list_physical_devices('GPU')
for gpu in gpus:
  tf.config.experimental.set_memory_growth(gpu, True)

2025-06-04 19:25:44.917676: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-06-04 19:25:44.917710: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-06-04 19:25:44.918781: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-04 19:25:44.924030: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-06-04 19:25:45.483052: W tensorflow/compiler/tf2

In [5]:
%load_ext autoreload
%autoreload 2

In [6]:
import numpy as np
from tqdm import trange
from UltimateTicTacToeEnvSelfPlay import UltimateTicTacToeEnvSelfPlay
from DoubleDQNAgent import DoubleDQNAgent
from MCTSImproved import MCTS

In [7]:
NUM_OF_GAMES = 100
envs = [UltimateTicTacToeEnvSelfPlay() for _ in range(NUM_OF_GAMES)]
state_space_shape = envs[0].to_state()[0].shape[0]
action_space_size = 81
ddqn_agent = DoubleDQNAgent(action_space_size, state_space_shape, loaded=True, model_path="../models/Thesis_DDQN_data_augmentation_2mln")
dones = np.zeros((NUM_OF_GAMES,), dtype=bool)
final_rewards = np.zeros((NUM_OF_GAMES,))
while(not np.all(dones) != 0):
    print(np.sum(dones), " out of ", len(dones))
    ddqn_states = np.array([env.to_state()[0] for env in envs])
    ddqn_available_actions = np.array([env.to_state()[1] for env in envs])
    ddqn_actions = ddqn_agent.choose_action(ddqn_states, ddqn_available_actions, True)
    ddqn_reward = np.zeros(NUM_OF_GAMES)
    game_finished = np.zeros(NUM_OF_GAMES)
    for i in range(NUM_OF_GAMES):
        _, ddqn_reward[i], game_finished[i], _  = envs[i].step(ddqn_actions[i])
        if game_finished[i] == True and dones[i] == False:
            dones[i] = True
            final_rewards[i] = ddqn_reward[i]
    
    mcts_agent = MCTS(envs, input_dim=state_space_shape, action_dim=action_space_size, simulations=1000)
    mcts_states = np.array([env.to_state()[0] for env in envs])
    mcts_available_actions = np.array([env.to_state()[1] for env in envs])
    mcts_actions = mcts_agent.play()
    mcts_reward = np.zeros(NUM_OF_GAMES)
    game_finished = np.zeros(NUM_OF_GAMES)
    for i in range(NUM_OF_GAMES):
            _, mcts_reward[i], game_finished[i], _  = envs[i].step(mcts_actions[i])
            if game_finished[i] == True and dones[i] == False:
                dones[i] = True
                final_rewards[i] = -mcts_reward[i]

    mcts_agent.update_tree_with_move(mcts_actions)
    print("Both players have done a move.")

win_as_first_player = np.count_nonzero(final_rewards == 1)/len(dones)
draw_as_first_player = np.count_nonzero(final_rewards == 0)/len(dones)

print("win rate as first player: ", win_as_first_player)
print("draw rate as first player: ", draw_as_first_player)

2025-06-04 19:25:46.584725: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-06-04 19:25:46.585036: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-06-04 19:25:46.585277: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

Models loaded from memory
0  out of  100


2025-06-04 19:25:47.842924: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8907


Both players have done a move.
0  out of  100
Both players have done a move.
0  out of  100
Both players have done a move.
0  out of  100
Both players have done a move.
0  out of  100
Both players have done a move.
0  out of  100
Both players have done a move.
0  out of  100
Both players have done a move.
0  out of  100
Both players have done a move.
0  out of  100
Both players have done a move.
0  out of  100
Both players have done a move.
0  out of  100
Both players have done a move.
0  out of  100
Both players have done a move.
0  out of  100
Both players have done a move.
0  out of  100
Both players have done a move.
1  out of  100
Both players have done a move.
5  out of  100
Both players have done a move.
10  out of  100
Both players have done a move.
12  out of  100
Both players have done a move.
15  out of  100
Both players have done a move.
22  out of  100
Both players have done a move.
30  out of  100
Both players have done a move.
38  out of  100
Both players have done a mov

In [8]:
NUM_OF_GAMES = 100
envs = [UltimateTicTacToeEnvSelfPlay() for _ in range(NUM_OF_GAMES)]
state_space_shape = envs[0].to_state()[0].shape[0]
action_space_size = 81
ddqn_agent = DoubleDQNAgent(action_space_size, state_space_shape, loaded=True, model_path="../models/Thesis_DDQN_data_augmentation_2mln")
mcts_agent = MCTS(envs, input_dim=state_space_shape, action_dim=action_space_size, simulations=1000)
dones = np.zeros((NUM_OF_GAMES,), dtype=bool)
final_rewards = np.zeros((NUM_OF_GAMES,))
while(not np.all(dones) != 0):
    print(np.sum(dones), " out of ", len(dones))
    mcts_states = np.array([env.to_state()[0] for env in envs])
    mcts_available_actions = np.array([env.to_state()[1] for env in envs])
    mcts_actions = mcts_agent.play()
    mcts_reward = np.zeros(NUM_OF_GAMES)
    game_finished = np.zeros(NUM_OF_GAMES)
    for i in range(NUM_OF_GAMES):
            _, mcts_reward[i], game_finished[i], _  = envs[i].step(mcts_actions[i])
            if game_finished[i] == True and dones[i] == False:
                dones[i] = True
                final_rewards[i] = -mcts_reward[i]
                
    ddqn_states = np.array([env.to_state()[0] for env in envs])
    ddqn_available_actions = np.array([env.to_state()[1] for env in envs])
    ddqn_actions = ddqn_agent.choose_action(ddqn_states, ddqn_available_actions, True)
    ddqn_reward = np.zeros(NUM_OF_GAMES)
    game_finished = np.zeros(NUM_OF_GAMES)
    for i in range(NUM_OF_GAMES):
        _, ddqn_reward[i], game_finished[i], _  = envs[i].step(ddqn_actions[i])
        if game_finished[i] == True and dones[i] == False:
            dones[i] = True
            final_rewards[i] = ddqn_reward[i]
    mcts_agent.update_tree_with_move(mcts_actions)
    mcts_agent.update_tree_with_move(ddqn_actions)
    print("Both players have done a move.")

win_as_second_player = np.count_nonzero(final_rewards == 1)/len(dones)
draw_as_second_player = np.count_nonzero(final_rewards == 0)/len(dones)

print("win rate as second player: ", win_as_second_player)
print("draw rate as second player: ", draw_as_second_player)

Models loaded from memory
0  out of  100
Both players have done a move.
0  out of  100
Both players have done a move.
0  out of  100
Both players have done a move.
0  out of  100
Both players have done a move.
0  out of  100
Both players have done a move.
0  out of  100
Both players have done a move.
0  out of  100
Both players have done a move.
0  out of  100
Both players have done a move.
0  out of  100
Both players have done a move.
0  out of  100
Both players have done a move.
0  out of  100
Both players have done a move.
0  out of  100
Both players have done a move.
0  out of  100
Both players have done a move.
0  out of  100
Both players have done a move.
0  out of  100
Both players have done a move.
1  out of  100
Both players have done a move.
4  out of  100
Both players have done a move.
8  out of  100
Both players have done a move.
15  out of  100
Both players have done a move.
23  out of  100
Both players have done a move.
27  out of  100
Both players have done a move.
36  o

In [9]:
total_win = (win_as_first_player+win_as_second_player)/2
total_draw = (draw_as_first_player+draw_as_second_player)/2
print("Total win rate: ", total_win)
print("Total draw rate: ", total_draw)
print("Total loss: ", 1-total_draw-total_win)

Total win rate:  0.7749999999999999
Total draw rate:  0.135
Total loss:  0.09000000000000008
